The code for pre-processing pictures of handwriting samples:
the data are taken from local folder, cut into patches with 3 lines of handwriting and saved to local path. Patches of each author are saved into separated folders

In [1]:
import os
from pathlib import Path
import cv2
import numpy as np
import os
import shutil
from pathlib import Path

data_path = "/content/"

In [2]:
def extract_line_patches(
    image_path: str,
    output_dir: str = "./patches",
    author_id: str = "unknown",
    lines_per_patch: int = 3,
    max_height: int =400
) -> int:
    '''Cuts images into patches with 3 lines of handwrting
    Args:
        image_path: path to image
        output_dir: the path to folder to save patches,
        author_id: id of author,
        lines_per_patch: number of lines per patch,
        max_height: max height of patch'''

    #reading the image from path
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0

    h, w = img.shape

    #Binarization
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Horizontal projection
    h_proj = np.sum(binary, axis=1) / 255

    # Searching for lines
    lines = []
    in_line = False
    start = 0
    noise_threshold = 50

    for y, val in enumerate(h_proj):
        if val > noise_threshold and not in_line:
            start = y
            in_line = True
        elif val <= noise_threshold and in_line:
            end = y
            if end - start > 15:  # minimal height of a line
                lines.append((start, end))
            in_line = False

    if in_line and h - start > 15:
        lines.append((start, h))

    # Filtering the lines. Use only line in the center of image
    valid_lines = []
    for start_y, end_y in lines:
        if start_y < h * 0.3 or end_y > h * 0.70:
            continue
        valid_lines.append((start_y, end_y))

    # Grouping lines into 3
    patches_coords = []
    for i in range(0, len(valid_lines), lines_per_patch):
        group = valid_lines[i:i+lines_per_patch]
        if not group:
            continue

        y_start = group[0][0]
        y_end = group[-1][1]

        padding = 10
        y_start = max(0, y_start - padding)
        y_end = min(h, y_end + padding)

        patches_coords.append((y_start, y_end))

    # Saving
    image_name = Path(image_path).stem
    author_output_dir = os.path.join(output_dir, author_id)
    os.makedirs(author_output_dir, exist_ok=True)

    saved_count = 0
    for idx, (y1, y2) in enumerate(patches_coords):
      if y2-y1>=max_height: #Saving patches with height >400
        patch = img[y1:y2, :]
        filename = f"{author_id}_{image_name}_patch{idx:03d}.png"
        filepath = os.path.join(author_output_dir, filename)
        cv2.imwrite(filepath, patch)
        saved_count += 1

    return saved_count

In [4]:
def process_all_authors(
    data_path: str,
    output_dir: str = None,
    lines_per_patch: int = 3,
    max_authors: int = None,
    max_height=400
):
    """
    Process all authors.

    Args:
        data_path: path to folder with images
        output_dir: the path to save patches
        lines_per_patch: the number of lines in 1 patch
        max_authors: max number of authors to process
        max_height: max height of a patch
    """
    # Define the input path
    if output_dir is None:
        output_dir = "/content/patches"

    # Geting all paths to authors' folders
    author_folders = []
    for item in os.listdir(data_path):
        item_path = os.path.join(data_path, item)
        if os.path.isdir(item_path) and item.isdigit():
            author_folders.append(item)

    author_folders.sort()
    if max_authors:
      author_folders = author_folders[:max_authors]
    total_images=0
    total_patches=0

    # Processing each author's folder
    for author_id in author_folders:
        author_path = os.path.join(data_path, author_id)

        # Getting all images of author
        image_files = []
        for file in os.listdir(author_path):
            if file.lower().endswith('.png'):
                image_files.append(file)

        image_files.sort()
        print(f"Images found: {len(image_files)}")

        author_images = 0
        author_patches = 0

        # Process each image
        for img_file in image_files:
            img_path = os.path.join(author_path, img_file)

            patches_count = extract_line_patches(
                image_path=img_path,
                output_dir=output_dir,
                author_id=author_id,
                lines_per_patch=lines_per_patch,
                max_height=max_height
            )

            if patches_count > 0:
                author_images += 1
                author_patches += patches_count

        if author_images > 0:
            print(f"Total images for author {author_id}: {author_images}, {author_patches} patches")
            total_images += author_images
            total_patches += author_patches
        else:
            print(f"No images for author {author_id}")


    print(f"Authors processed: {len(author_folders)}")
    print(f"Images processed: {total_images}")
    print(f"Total patches: {total_patches}")


In [11]:
content = os.listdir(data_path)
if "data" in content and os.path.isdir(os.path.join(data_path, "data")):
  data_path = os.path.join(data_path, "data")
  author_dirs = [d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d)) and d.isdigit()]
  process_all_authors(data_path=data_path, output_dir="/content/patches",  lines_per_patch=3,max_authors=None)

Images found: 59
Total images for author 000: 58, 115 patches
Images found: 2
Total images for author 001: 2, 3 patches
Images found: 1
Total images for author 002: 1, 2 patches
Images found: 2
Total images for author 003: 2, 4 patches
Images found: 1
Total images for author 004: 1, 2 patches
Images found: 2
Total images for author 005: 2, 3 patches
Images found: 1
No images for author 006
Images found: 2
Total images for author 007: 2, 4 patches
Images found: 2
Total images for author 008: 2, 4 patches
Images found: 2
Total images for author 009: 2, 4 patches
Images found: 2
Total images for author 010: 2, 4 patches
Images found: 2
Total images for author 011: 2, 3 patches
Images found: 2
Total images for author 012: 2, 4 patches
Images found: 3
Total images for author 013: 3, 6 patches
Images found: 2
Total images for author 014: 2, 4 patches
Images found: 1
Total images for author 015: 1, 2 patches
Images found: 3
Total images for author 016: 3, 6 patches
Images found: 3
Total image

Now let's process images into 1 line per patch and save in separate folder

In [10]:
data_path = "/content/"
content = os.listdir(data_path)
if "data" in content and os.path.isdir(os.path.join(data_path, "data")):
  data_path = os.path.join(data_path, "data")
  author_dirs = [d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d)) and d.isdigit()]
  process_all_authors(data_path=data_path, output_dir="/content/patches_one_line",  lines_per_patch=1,max_authors=None, max_height=50)

Images found: 59
Total images for author 000: 58, 424 patches
Images found: 2
Total images for author 001: 2, 11 patches
Images found: 1
Total images for author 002: 1, 8 patches
Images found: 2
Total images for author 003: 2, 15 patches
Images found: 1
Total images for author 004: 1, 7 patches
Images found: 2
Total images for author 005: 2, 11 patches
Images found: 1
Total images for author 006: 1, 5 patches
Images found: 2
Total images for author 007: 2, 16 patches
Images found: 2
Total images for author 008: 2, 16 patches
Images found: 2
Total images for author 009: 2, 16 patches
Images found: 2
Total images for author 010: 2, 16 patches
Images found: 2
Total images for author 011: 2, 14 patches
Images found: 2
Total images for author 012: 2, 16 patches
Images found: 3
Total images for author 013: 3, 20 patches
Images found: 2
Total images for author 014: 2, 16 patches
Images found: 1
Total images for author 015: 1, 7 patches
Images found: 3
Total images for author 016: 3, 24 patche